In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from src.dataset import SaltBaseDataset
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
df = load_dataset(
    "Vikhrmodels/ToneBooksPlus_quantized-bigcodec"
)

In [4]:
df["train"][7]

{'text': 'В своём тулупе, меховой шапке и с бородой, покрытой инеем, он казался настоящим Дедом Морозом.',
 'voice_name': 'aleksandra_nizhegorodova',
 'audio_tokens': [2550,
  2076,
  692,
  1915,
  1176,
  1176,
  1176,
  1176,
  1176,
  1176,
  1176,
  1176,
  1176,
  1176,
  1176,
  1915,
  692,
  7239,
  3550,
  7853,
  2574,
  4475,
  2265,
  7240,
  2023,
  4261,
  1884,
  344,
  2249,
  107,
  3459,
  625,
  1714,
  4635,
  3295,
  5901,
  5980,
  5392,
  7703,
  1871,
  3911,
  6353,
  369,
  8070,
  3018,
  619,
  6885,
  1631,
  7045,
  5360,
  6174,
  1054,
  1339,
  7099,
  5223,
  450,
  1295,
  6555,
  7937,
  2516,
  2055,
  511,
  1678,
  3815,
  6581,
  7225,
  1188,
  1442,
  5770,
  4542,
  852,
  2594,
  4338,
  1413,
  5961,
  7135,
  7117,
  546,
  4628,
  2528,
  4618,
  1070,
  1669,
  3376,
  7407,
  6742,
  2535,
  575,
  225,
  5324,
  1472,
  2440,
  3674,
  4733,
  2316,
  6556,
  8047,
  2519,
  444,
  7021,
  4661,
  6497,
  7748,
  4985,
  2320,
  4345,


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-270m-it",
    max_seq_length=4096,
    dtype=torch.bfloat16,
    full_finetuning=True,
    device_map="cuda:1"
)

Unsloth: You selected full finetuning support, but 4bit / 8bit is enabled - disabling LoRA / QLoRA.
==((====))==  Unsloth 2025.12.5: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 2. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


In [6]:
audio_special_tokens = [f"<|bigcodec_{i}|>" for i in range(8192)] 
tokenizer.add_tokens(audio_special_tokens)
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Gemma3TextScaledWordEmbedding(270337, 640, padding_idx=0)

In [7]:
dataset = SaltBaseDataset(tokenizer=tokenizer, hf_dataset=df["train"])

In [8]:
next(iter(dataset))

{'input_ids': tensor([   2,  105, 2364,  ...,    0,    0,    0]),
 'attention_mask': tensor([1, 1, 1,  ..., 0, 0, 0]),
 'labels': tensor([-100, -100, -100,  ..., -100, -100, -100])}

In [9]:
tokenizer.decode(next(iter(dataset))["input_ids"])

'<bos><start_of_turn>user\nВы полезный помощник по озвучиванию текста\n\nПреобразуй это в аудио: Сорокапятилетний Немцов был худощавым меланхоличным мужчиной, неинициативным, излишне осторожным.<end_of_turn>\n<start_of_turn>model\n<|bigcodec_2550|> <|bigcodec_2076|> <|bigcodec_692|> <|bigcodec_692|> <|bigcodec_692|> <|bigcodec_1915|> <|bigcodec_692|> <|bigcodec_4668|> <|bigcodec_692|> <|bigcodec_4668|> <|bigcodec_4668|> <|bigcodec_4668|> <|bigcodec_4668|> <|bigcodec_692|> <|bigcodec_4668|> <|bigcodec_5755|> <|bigcodec_2394|> <|bigcodec_6042|> <|bigcodec_5320|> <|bigcodec_7430|> <|bigcodec_7290|> <|bigcodec_5068|> <|bigcodec_3074|> <|bigcodec_6741|> <|bigcodec_3143|> <|bigcodec_3465|> <|bigcodec_327|> <|bigcodec_873|> <|bigcodec_4686|> <|bigcodec_3188|> <|bigcodec_6701|> <|bigcodec_3886|> <|bigcodec_93|> <|bigcodec_3369|> <|bigcodec_7355|> <|bigcodec_2569|> <|bigcodec_2479|> <|bigcodec_393|> <|bigcodec_6215|> <|bigcodec_70|> <|bigcodec_4641|> <|bigcodec_3608|> <|bigcodec_7211|> <|bigcod

In [10]:
model.resize_token_embeddings(len(tokenizer))

Gemma3TextScaledWordEmbedding(270337, 640, padding_idx=0)

In [11]:
tokenizer.special_tokens_map

{'bos_token': '<bos>',
 'eos_token': '<end_of_turn>',
 'unk_token': '<unk>',
 'pad_token': '<pad>',
 'boi_token': '<start_of_image>',
 'eoi_token': '<end_of_image>',
 'image_token': '<image_soft_token>'}

In [12]:
# 1. Получаем один sample
sample = next(iter(dataset))

# 2. Смотрим что внутри
print(sample.keys())
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"labels shape: {sample['labels'].shape}")

# 3. Декодируем input_ids (полный текст)
print("\n=== Full text ===")
print(tokenizer.decode(sample["input_ids"]))

# 4. Декодируем только trainable часть (где labels != -100)
print("\n=== Trainable part (labels != -100) ===")
trainable_ids = sample["input_ids"][sample["labels"] != -100]
print(tokenizer.decode(trainable_ids))

# 5. Статистика
total = len(sample["input_ids"])
masked = (sample["labels"] == -100).sum().item()
trainable = (sample["labels"] != -100).sum().item()

print(f"\n=== Stats ===")
print(f"Total: {total}, Masked: {masked}, Trainable: {trainable}")


dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([2560])
labels shape: torch.Size([2560])

=== Full text ===
<bos><start_of_turn>user
Вы полезный помощник по озвучиванию текста

Преобразуй это в аудио: Сорокапятилетний Немцов был худощавым меланхоличным мужчиной, неинициативным, излишне осторожным.<end_of_turn>
<start_of_turn>model
<|bigcodec_2550|> <|bigcodec_2076|> <|bigcodec_692|> <|bigcodec_692|> <|bigcodec_692|> <|bigcodec_1915|> <|bigcodec_692|> <|bigcodec_4668|> <|bigcodec_692|> <|bigcodec_4668|> <|bigcodec_4668|> <|bigcodec_4668|> <|bigcodec_4668|> <|bigcodec_692|> <|bigcodec_4668|> <|bigcodec_5755|> <|bigcodec_2394|> <|bigcodec_6042|> <|bigcodec_5320|> <|bigcodec_7430|> <|bigcodec_7290|> <|bigcodec_5068|> <|bigcodec_3074|> <|bigcodec_6741|> <|bigcodec_3143|> <|bigcodec_3465|> <|bigcodec_327|> <|bigcodec_873|> <|bigcodec_4686|> <|bigcodec_3188|> <|bigcodec_6701|> <|bigcodec_3886|> <|bigcodec_93|> <|bigcodec_3369|> <|bigcodec_7355|> <|bigcodec_256